In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
import csv
import pandas as pd

In [6]:
# The way this part work, is to run it, then scan the pop-up. Don't close the pop up yet, click enter in the python response. Continue run the rest of the code
driver = webdriver.Chrome()  
driver.get("https://web.whatsapp.com")
input("Scan the QR code and press Enter...") 

Scan the QR code and press Enter... 


''

In [7]:
# Test Data
# Your contact list
contacts = [
    {"name": "testwrongnumber", "number": "62123123123"},
    {"name": "Iqbal", "number": "6281330469005"},
]

# Your message template
template = """Selamat Pagi, Bapak/Ibu {name}%0A
Perkenalkan saya Melinda Wijaya dari liveaman.com
Kami menawarkan benefit karyawan dalam bentuk pengelolaan Asuransi Kesehatan, paket MCU, Teleconsultation, dan juga Mental Health Care.%0A
Apakah Bapak/Ibu bersedia untuk memberikan waktu untuk melakukan perkenalan singkat? Terima kasih 🙏"""


In [8]:
# To collect results
results = []

# Send messages
for contact in contacts:
    name = contact["name"]
    number = contact["number"]
    message = template.format(name=name)
    message_encoded = message.replace(" ", "%20").replace("\n", "%0A")
    url = f"https://web.whatsapp.com/send?phone={number}&text={message_encoded}"
    
    driver.get(url)
    time.sleep(10)  # Allow chat to load

    try:
        send_button = driver.find_element(By.XPATH, '//span[@data-icon="send"]')
        send_button.click()
        print(f"Sent to {name} ({number})")
        results.append({"name": name, "number": number, "status": "Success"})
    except Exception as e:
        print(f"Failed to send to {name} ({number}): {e}")
        results.append({"name": name, "number": number, "status": "Failed", "error": str(e)})
    
    time.sleep(5)

driver.quit()

Failed to send to testwrongnumber (62123123123): Message: no such element: Unable to locate element: {"method":"xpath","selector":"//span[@data-icon="send"]"}
  (Session info: chrome=135.0.7049.116); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
0   chromedriver                        0x000000010dcc92c8 chromedriver + 6197960
1   chromedriver                        0x000000010dcc08ea chromedriver + 6162666
2   chromedriver                        0x000000010d745de0 chromedriver + 417248
3   chromedriver                        0x000000010d797797 chromedriver + 751511
4   chromedriver                        0x000000010d7979b1 chromedriver + 752049
5   chromedriver                        0x000000010d7e79b4 chromedriver + 1079732
6   chromedriver                        0x000000010d7bd9ed chromedriver + 907757
7   chromedriver                        0x000000010d7e4cdb chromedriver +

In [12]:
# Save to Excel
df = pd.DataFrame(results)
excel_file = "wa_send_results.xlsx"
df.to_excel(excel_file, index=False)

print(f"Results saved to {excel_file}")

Results saved to wa_send_results.xlsx


In [9]:
# Write results to CSV
csv_file = "wa_send_results.csv"
with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
    fieldnames = ["name", "number", "status", "error"]
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    for result in results:
        writer.writerow(result)

print(f"Results saved to {csv_file}")

Results saved to wa_send_results.csv
